# ★ Mini-Capstone 1 — Scientific Paper Assistant

**Goal:** Integrate Modules 1 + 2 + 3.  **Time:** ~4 hours

Build: Pydantic extraction → ChromaDB + Multi-Query RAG → tracked conversation

In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print('✅ llm_checker loaded')

In [ ]:
import importlib
REQUIRED = ['openai','instructor','pydantic','tiktoken','chromadb','sentence_transformers','datasets']
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
print('Missing:', missing if missing else 'None ✅')

In [ ]:
import pandas as pd
DATA_PATH = os.path.join('data', 'arxiv', 'arxiv_abstracts_5k.csv')
if os.path.exists(DATA_PATH):
    arxiv_df = pd.read_csv(DATA_PATH)
    print(f'✅ Loaded {len(arxiv_df)} rows')
else:
    try:
        from datasets import load_dataset
        ds = load_dataset('gfissore/arxiv-abstracts-2021', split='train').select(range(500))
        arxiv_df = ds.to_pandas()
        os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
        arxiv_df.to_csv(DATA_PATH, index=False)
        print(f'Downloaded {len(arxiv_df)} rows')
    except Exception as e:
        print(f'⚠️  Run setup_datasets.py first: {e}')
        arxiv_df = pd.DataFrame()

---
## Step 1 — Paper Ingestion Pipeline (M2 + M3 skill)

Extract structured metadata from 100 arXiv abstracts using Pydantic + Instructor.

In [ ]:
show_exercise(
    'MC1-1', 'Paper ingestion pipeline — Pydantic extraction',
    'Define PaperMetadata(BaseModel): title, topic_area, key_concepts (max 5), year_approx. '
    'Use Instructor + LM Studio to extract from each abstract. Target pass_rate >= 90%.',
    hints=[
        'instructor.patch(client) wraps the OpenAI client',
        'topic_area: LLM-inferred field e.g. NLP, Computer Vision',
        'Catch exceptions per row to avoid aborting the whole batch',
    ],
    checks=[
        'PaperMetadata has >= 4 typed fields',
        'extract_metadata(abstract) -> PaperMetadata | None',
        'pass_rate >= 0.90 across 100 abstracts',
    ],
)

In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional, List
import instructor

# LM Studio client — free local model
lm_client = instructor.patch(OpenAI(
    base_url='http://localhost:1234/v1', api_key='lm-studio'
))


# TODO: define your Pydantic model
class PaperMetadata(BaseModel):
    title: str = Field(description='Inferred paper title')
    topic_area: str = Field(description="E.g. 'NLP', 'CV', 'RL'")
    key_concepts: List[str] = Field(default_factory=list)
    year_approx: Optional[int] = None


# TODO: implement extraction
def extract_metadata(abstract: str) -> Optional[PaperMetadata]:
    # Use lm_client.chat.completions.create(..., response_model=PaperMetadata)
    pass

In [ ]:
_sample = (arxiv_df.iloc[0].get('abstract', 'Transformers for NLP.')
           if len(arxiv_df) else 'Transformers for NLP.')
try:
    _r = extract_metadata(str(_sample)[:1500])
except Exception as _e:
    _r = None; print(f'extract_metadata raised: {_e}')

check([
    (issubclass(PaperMetadata, BaseModel), 'PaperMetadata is a BaseModel'),
    (len(PaperMetadata.model_fields) >= 4, 'At least 4 fields'),
    ('title' in PaperMetadata.model_fields, 'Field title exists'),
    ('key_concepts' in PaperMetadata.model_fields, 'Field key_concepts exists'),
    (_r is not None, 'extract_metadata() returns a result'),
    (isinstance(_r, PaperMetadata) if _r else False, 'Returns PaperMetadata'),
], exercise_id='MC1-1')

In [ ]:
sample_df = arxiv_df.head(100)
results, failures = [], []

for i, row in enumerate(sample_df.itertuples()):
    abstract = str(getattr(row, 'abstract', getattr(row, 'text', '')))[:1500]
    try:
        meta = extract_metadata(abstract)
        if meta: results.append(meta)
        else: failures.append(i)
    except Exception: failures.append(i)

pass_rate = len(results) / max(len(sample_df), 1)
print(f'Extracted {len(results)}/100  pass_rate = {pass_rate:.2%}')
check([
    (pass_rate >= 0.90, f'Pass rate >= 90% (got {pass_rate:.1%})'),
    (len(results) >= 90, 'At least 90 successful extractions'),
], exercise_id='MC1-1b')

In [ ]:
evaluate(extract_metadata,
         'Extract PaperMetadata from arXiv abstracts using Instructor. Target 90% pass rate.',
         exercise_id='MC1-1')

---
## Step 2 — Multi-Query RAG Answer (M3 skill)

Index papers in ChromaDB, answer using 3 query variants + RRF.

In [ ]:
show_exercise(
    'MC1-2', 'Multi-Query RAG over indexed papers',
    'Index 100 papers in ChromaDB. Implement multi_query_rag(question) -> dict '
    'with keys: answer, sources, num_queries_generated (must be 3). '
    'Use RRF or union to merge retrieval results.',
    hints=[
        "SentenceTransformerEmbeddingFunction('all-MiniLM-L6-v2') — no API key",
        'Generate 3 query variants with a quick LLM prompt',
        'RRF score = sum(1 / (60 + rank_i)) across queries',
    ],
    checks=[
        'ChromaDB collection has >= 90 documents',
        'multi_query_rag() returns dict with answer + sources',
        'num_queries_generated == 3',
    ],
)

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

emb_fn = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
chroma_client = chromadb.Client()
try: chroma_client.delete_collection('mc1_arxiv')
except Exception: pass
collection = chroma_client.create_collection('mc1_arxiv', embedding_function=emb_fn)

for i, meta in enumerate(results):
    doc = f'Title: {meta.title}. Area: {meta.topic_area}. Concepts: {", ".join(meta.key_concepts)}'
    collection.add(documents=[doc],
                   metadatas=[{'title': meta.title, 'topic': meta.topic_area}],
                   ids=[f'paper_{i}'])

print(f'✅ Indexed {collection.count()} papers')


def _query_variants(question: str) -> list:
    raw = OpenAI(base_url='http://localhost:1234/v1', api_key='lm-studio')
    prompt = f"Generate 3 short search queries to find papers about: '{question}'.\nReturn ONLY 3 lines."
    resp = raw.chat.completions.create(
        model='local-model',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=120, temperature=0.5,
    )
    lines = [l.strip('- 123.').strip() for l in resp.choices[0].message.content.strip().split('\n') if l.strip()]
    return lines[:3]


def multi_query_rag(question: str) -> dict:
    # TODO: generate 3 variants, query ChromaDB for each, merge, generate answer
    pass

In [ ]:
_q = 'What are recent advances in attention mechanisms?'
try: _out = multi_query_rag(_q)
except Exception as _e: _out = None; print(f'Error: {_e}')

check([
    (collection.count() >= 90, f'ChromaDB docs: {collection.count()}'),
    (_out is not None, 'multi_query_rag() returned a result'),
    (isinstance(_out, dict) if _out else False, 'Returns dict'),
    ('answer' in (_out or {}), 'answer key present'),
    ('sources' in (_out or {}), 'sources key present'),
    ((_out or {}).get('num_queries_generated', 0) == 3, '3 query variants generated'),
], exercise_id='MC1-2')

In [ ]:
evaluate(multi_query_rag, 'Multi-Query RAG: 3 variants, ChromaDB, RRF merge, LLM answer.', exercise_id='MC1-2')

---
## Step 3 — Context Window + Cost Tracking (M1 skill)

5-turn conversation keeping context ≤ 3000 tokens. Track cost per turn.

In [ ]:
show_exercise(
    'MC1-3', '5-turn conversation with context & cost tracking',
    'Wrap multi_query_rag in a loop: maintain messages list, trim to <=3000 tokens, '
    'track cost (gpt-4o-mini: $0.15/1M input, $0.60/1M output). '
    'Return conversation_log: list of {turn, question, answer, tokens_used, cost_usd}.',
    hints=[
        "tiktoken.encoding_for_model('gpt-4o-mini') to count tokens",
        'Pop oldest user+assistant pair when token budget exceeded',
        'For LM Studio: set cost = 0.0 (it is free)',
    ],
    checks=[
        'conversation_log has 5 entries',
        'All turns have tokens_used <= 3000',
        'cost_usd tracked per turn',
    ],
)

In [ ]:
import tiktoken

ENCODING = tiktoken.encoding_for_model('gpt-4o-mini')
MAX_TOKENS = 3000

def count_tokens(messages):
    return sum(len(ENCODING.encode(m.get('content', '') or '')) for m in messages)


def chat_with_rag(questions: list) -> list:
    messages = [{'role': 'system', 'content': 'You are a scientific paper assistant.'}]
    log = []
    raw_client = OpenAI(base_url='http://localhost:1234/v1', api_key='lm-studio')

    for turn, question in enumerate(questions, 1):
        # TODO: get RAG context, build user message, trim, call LLM, log
        pass

    return log


QUESTIONS = [
    'What neural architectures are used for NLP tasks?',
    'How does attention compare to RNN approaches?',
    'Which papers discuss BERT or GPT?',
    'What are the key limitations mentioned?',
    'Summarise our transformer discussion.',
]
conversation_log = chat_with_rag(QUESTIONS)
print(f'Turns: {len(conversation_log)}')
for e in conversation_log:
    print(f"  Turn {e.get('turn','?')}: {e.get('tokens_used','?')} tokens | ${e.get('cost_usd',0):.5f}")

In [ ]:
check([
    (isinstance(conversation_log, list), 'Returns a list'),
    (len(conversation_log) == 5, f'5 turns logged (got {len(conversation_log)})'),
    (all('tokens_used' in e for e in conversation_log), 'tokens_used tracked'),
    (all(e.get('tokens_used', 9999) <= 3000 for e in conversation_log), 'All turns <= 3000 tokens'),
    (all('cost_usd' in e for e in conversation_log), 'cost_usd present'),
], exercise_id='MC1-3')

---
## Step 4 — Full Evaluation + Report (🔴 CHALLENGE)

10 test questions, LLM-as-judge scoring, markdown report.

In [ ]:
show_exercise(
    'MC1-4', 'Evaluate pipeline + write report',
    'Run 10 test questions through multi_query_rag. '
    'Use LLM-as-judge to score each answer 1-5 (relevance + groundedness). '
    'Average score must be >= 3.0. Write mini_capstone_1_report.md.',
    hints=[
        "Judge prompt: 'Score 1-5 (integer only): relevant, grounded, complete?'",
        "Parse score: re.findall(r'\\\\d', judge_output)[0]",
        'LM Studio cost = $0.00',
    ],
    checks=[
        '10 questions evaluated',
        'avg_judge_score >= 3.0',
        'mini_capstone_1_report.md written',
    ],
)

In [ ]:
import re, statistics

TEST_QUESTIONS = [
    'What neural architectures are used for NLP?',
    'How are graph neural networks applied?',
    'Which papers address model efficiency?',
    'What evaluation metrics are most common?',
    'How is self-supervised learning used?',
    'Which datasets are commonly used?',
    'What future directions are mentioned?',
    'How do papers approach multi-modal learning?',
    'Are there healthcare applications?',
    'What are the main training techniques?',
]

JUDGE_SYS = 'Score 1-5 (integer only). 1=irrelevant, 5=relevant+grounded+complete.'

def judge_answer(question, answer, sources):
    raw = OpenAI(base_url='http://localhost:1234/v1', api_key='lm-studio')
    srcs = chr(10).join(f'- {s}' for s in (sources or [])[:5])
    prompt = f'Question: {question}\nSources:\n{srcs}\nAnswer: {answer}\nScore (1-5):'
    resp = raw.chat.completions.create(
        model='local-model',
        messages=[{'role':'system','content':JUDGE_SYS},{'role':'user','content':prompt}],
        max_tokens=5, temperature=0,
    )
    nums = re.findall(r'\d', resp.choices[0].message.content.strip())
    return int(nums[0]) if nums else 3

eval_results = []
for q in TEST_QUESTIONS:
    out = multi_query_rag(q) or {}
    score = judge_answer(q, out.get('answer',''), out.get('sources',[]))
    eval_results.append({'question': q, 'score': score})
    print(f'  [{score}/5] {q[:60]}')

avg_score = statistics.mean(r['score'] for r in eval_results)
print(f'\n📊 Average: {avg_score:.2f}/5.0')
check([
    (len(eval_results) == 10, '10 Q&A pairs evaluated'),
    (avg_score >= 3.0, f'Avg >= 3.0 (got {avg_score:.2f})'),
], exercise_id='MC1-4')

In [ ]:
total_cost = sum(e.get('cost_usd', 0) for e in conversation_log)
lines = [f"| {r['question'][:50]} | {r['score']}/5 |" for r in eval_results]
report_md = '\n'.join([
    '# Mini-Capstone 1 Report — Scientific Paper Assistant',
    '',
    '## Metrics',
    '| Metric | Value | Target | OK |',
    '|--------|-------|--------|----|',
    f'| Extraction pass rate | {pass_rate:.1%} | ≥ 90% | {"✅" if pass_rate>=0.90 else "❌"} |',
    f'| ChromaDB docs | {collection.count()} | ≥ 90 | {"✅" if collection.count()>=90 else "❌"} |',
    f'| LLM-as-judge avg | {avg_score:.2f} | ≥ 3.0 | {"✅" if avg_score>=3.0 else "❌"} |',
    '',
    '## Cost Comparison (5-turn conversation)',
    '| Backend | Cost |',
    '|---------|------|',
    f'| GPT-4o-mini | ${total_cost:.4f} |',
    '| LM Studio (local) | $0.0000 |',
    '',
    '## Judge Scores',
    '| Question | Score |',
    '|----------|-------|',
] + lines)
with open('mini_capstone_1_report.md', 'w') as f:
    f.write(report_md)
print('✅ mini_capstone_1_report.md written')

In [ ]:
check([
    (pass_rate >= 0.90, f'Extraction pass rate >= 90%'),
    (collection.count() >= 90, 'ChromaDB >= 90 docs'),
    (len(conversation_log) == 5, '5-turn conversation complete'),
    (all(e.get('tokens_used',9999)<=3000 for e in conversation_log), 'Context <= 3000 tokens'),
    (avg_score >= 3.0, f'Judge score {avg_score:.2f} >= 3.0'),
    (os.path.exists('mini_capstone_1_report.md'), 'Report written'),
], exercise_id='MC1-final')
progress()